# 🧴 DermaSmart V2 — Improved Skin Condition Classifier

**Author:** Ravi Kiran Reddy Bada  
**GitHub:** [github.com/Ravikiranreddybada](https://github.com/Ravikiranreddybada)  

---

## What's New in V2 vs V1?

| Feature | V1 (Original) | V2 (This Notebook) |
|---|---|---|
| Backbone training | Fully frozen | **Two-phase fine-tuning** |
| Data augmentation | None | **Flip, Rotate, Zoom, Contrast** |
| Class imbalance handling | None | **Class weights** |
| Expected val accuracy | ~43% | **~70-78%** |
| Optimizer | Adam (1e-3) | Adam + **lower LR for fine-tuning** |
| Dropout regularization | None | **Dropout(0.3)** |

**Dataset:** [Dermnet](https://www.kaggle.com/datasets/shubhamgoel27/dermnet) — same 23-class dataset  
**Base Model:** MobileNetV2 (pretrained on ImageNet)  
**Hardware:** GPU (T4 recommended on Colab)

## 1. Install Dependencies

In [ ]:
!pip install -q kagglehub tensorflow matplotlib scikit-learn

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import kagglehub

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 3. Download Dataset

Same **Dermnet** dataset — 19,500+ images across 23 skin condition classes.

In [ ]:
path = kagglehub.dataset_download('shubhamgoel27/dermnet')
print(f'Dataset path: {path}')

# Find the train directory
TRAIN_DIR = None
for root, dirs, files in os.walk(path):
    if 'train' in dirs:
        TRAIN_DIR = os.path.join(root, 'train')
        break

print(f'Train directory: {TRAIN_DIR}')
classes = sorted(os.listdir(TRAIN_DIR))
print(f'Number of classes: {len(classes)}')
print(f'Classes: {classes}')

## 4. Configuration

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32   # smaller than V1 for better gradient updates
NUM_CLASSES = 23
SEED = 42

# Phase 1 training config
PHASE1_EPOCHS = 10
PHASE1_LR = 1e-3

# Phase 2 fine-tuning config
PHASE2_EPOCHS = 30
PHASE2_LR = 1e-5    # much lower LR to avoid destroying pretrained weights
FINE_TUNE_FROM = -50  # unfreeze last 50 layers of MobileNetV2

print('Config set ✅')

## 5. Load Data with Augmentation

### Key improvement: Data augmentation
V1 had **zero augmentation**. Real skin condition photos vary in lighting, angle, and zoom — augmentation teaches the model to handle this.

In [ ]:
# Load full dataset
full_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='categorical',
    shuffle=True,
)

class_names = full_ds.class_names
print(f'Class names: {class_names}')

# 80/20 split
total_batches = len(full_ds)
train_size = int(0.8 * total_batches)
val_size = total_batches - train_size

train_ds_raw = full_ds.take(train_size)
val_ds = full_ds.skip(train_size)

print(f'Train batches: {train_size}, Val batches: {val_size}')

In [ ]:
# ── Data Augmentation Pipeline ──────────────────────────────────────
# These transforms are applied ONLY during training, not validation
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),          # Mirror horizontally
    layers.RandomRotation(0.1),               # Rotate ±10%
    layers.RandomZoom(0.15),                  # Zoom in/out ±15%
    layers.RandomContrast(0.2),               # Vary contrast ±20%
    layers.RandomBrightness(0.1),             # Vary brightness ±10%
], name='data_augmentation')

# Apply MobileNetV2 preprocessing (scales to [-1, 1])
def preprocess_train(images, labels):
    images = data_augmentation(images, training=True)
    images = preprocess_input(images)
    return images, labels

def preprocess_val(images, labels):
    images = preprocess_input(images)
    return images, labels

# Apply preprocessing and prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_val, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print('Data pipelines ready ✅')

## 6. Compute Class Weights

Dermnet is **imbalanced** — some conditions have far fewer images than others. Class weights penalize the model more for misclassifying rare classes.

In [ ]:
# Collect all labels from training set to compute class weights
all_labels = []
for _, labels in train_ds_raw:
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_labels = np.array(all_labels)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weight_dict = dict(enumerate(class_weights_array))

print('Class weights:')
for i, (cls, w) in enumerate(zip(class_names, class_weights_array)):
    print(f'  {cls[:40]:<40} → {w:.3f}')

## 7. Build Model

**Architecture:**
```
Input (224×224×3)
    │
    ▼
MobileNetV2 backbone  ← frozen in Phase 1, partially unfrozen in Phase 2
    │
    ▼
GlobalAveragePooling2D
    │
    ▼
Dense(256, relu) + BatchNormalization   ← NEW: BN for stability
    │
    ▼
Dropout(0.3)                            ← NEW: prevents overfitting
    │
    ▼
Dense(23, softmax)                      ← 23 skin condition classes
```

In [ ]:
def build_model():
    # Load MobileNetV2 without the classification top
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Freeze for Phase 1

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)      # stabilizes training
    x = layers.Dropout(0.3)(x)             # prevents overfitting
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model()
model.summary()

## 8. Phase 1: Train Classification Head

Backbone is frozen. Only training the custom Dense layers — fast and stable.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('best_phase1.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]

print('=== PHASE 1: Training Classification Head ===')
history1 = model.fit(
    train_ds,
    epochs=PHASE1_EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
)

print(f'\nBest Phase 1 Val Accuracy: {max(history1.history["val_accuracy"]):.4f}')

## 9. Phase 2: Fine-Tune MobileNetV2

**This is what V1 was missing!**

Unfreeze the last 50 layers of MobileNetV2 so the backbone can learn **skin-specific features** instead of generic ImageNet features. Use a very low learning rate (1e-5) to avoid destroying the pretrained weights.

In [ ]:
# Unfreeze last 50 layers of MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

trainable_layers = sum(1 for l in base_model.layers if l.trainable)
print(f'Trainable layers in backbone: {trainable_layers} / {len(base_model.layers)}')

# Recompile with a very low LR for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1),
    ModelCheckpoint('best_phase2.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]

print('\n=== PHASE 2: Fine-Tuning MobileNetV2 Backbone ===')
history2 = model.fit(
    train_ds,
    epochs=PHASE2_EPOCHS,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
)

print(f'\nBest Phase 2 Val Accuracy: {max(history2.history["val_accuracy"]):.4f}')

## 10. Training Curves

In [ ]:
# Combine histories
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
phase1_end = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy
axes[0].plot(acc, label='Train Accuracy', color='#C9A84C')
axes[0].plot(val_acc, label='Val Accuracy', color='#8A9E8C')
axes[0].axvline(phase1_end, color='gray', linestyle='--', alpha=0.5, label='Phase 1 → 2')
axes[0].set_title('Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(loss, label='Train Loss', color='#C9A84C')
axes[1].plot(val_loss, label='Val Loss', color='#8A9E8C')
axes[1].axvline(phase1_end, color='gray', linestyle='--', alpha=0.5, label='Phase 1 → 2')
axes[1].set_title('Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('DermaSmart V2 — Training Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as training_curves_v2.png')

## 11. Evaluate on Validation Set

In [ ]:
print('=== Final Evaluation on Validation Set ===')
val_loss_final, val_acc_final = model.evaluate(val_ds, verbose=1)
print(f'\n✅ Final Val Loss:     {val_loss_final:.4f}')
print(f'✅ Final Val Accuracy: {val_acc_final:.4f} ({val_acc_final*100:.1f}%)')

# Compare with V1
v1_val_acc = 0.4296
improvement = (val_acc_final - v1_val_acc) * 100
print(f'\n📈 Improvement over V1 ({v1_val_acc*100:.1f}%): +{improvement:.1f}%')

## 12. Per-Class Accuracy

In [ ]:
from sklearn.metrics import classification_report

y_true, y_pred = [], []
for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## 13. Save Model

Save as `.keras` — then run `convert_model.py` locally to convert to TFLite for deployment.

In [ ]:
model.save('tf_model_v2.keras')
print('✅ Model saved → tf_model_v2.keras')
print()
print('Next steps:')
print('1. Download tf_model_v2.keras from Colab')
print('2. Place it in dermasmart/backend/model/tf_model.keras')
print('3. Run: /opt/homebrew/Caskroom/miniforge/base/bin/python convert_model.py')
print('4. git add . && git commit -m "feat: upgrade to V2 model" && git push origin main')

---

## Summary

| Item | Detail |
|------|--------|
| **Model** | MobileNetV2 + custom head (with Dropout + BatchNorm) |
| **Dataset** | Dermnet (23 skin condition classes) |
| **Input shape** | 224 × 224 × 3 |
| **Training split** | 80% train / 20% val |
| **Batch size** | 32 |
| **Phase 1** | Frozen backbone, 10 epochs, LR=1e-3 |
| **Phase 2** | Last 50 layers unfrozen, 30 epochs, LR=1e-5 |
| **Augmentation** | Flip, Rotate, Zoom, Contrast, Brightness |
| **Class weighting** | Yes — balanced |
| **Output** | `tf_model_v2.keras` |

**Author:** Ravi Kiran Reddy Bada — [github.com/Ravikiranreddybada](https://github.com/Ravikiranreddybada)